In [18]:
from datasets import load_from_disk
from collections import Counter
import plotly.express as px
import pandas as pd

tokenized = load_from_disk("../data/processed/tokenized_dataset")

train_ds = tokenized["train"]
val_ds   = tokenized["validation"]

print(train_ds)
print(val_ds)


Dataset({
    features: ['labels', 'doc_id', 'input_ids', 'attention_mask', 'special_tokens_mask', 'offset_mapping'],
    num_rows: 5779
})
Dataset({
    features: ['labels', 'doc_id', 'input_ids', 'attention_mask', 'special_tokens_mask', 'offset_mapping'],
    num_rows: 1282
})


In [19]:
# zbieramy doc_id z obu splitów
all_doc_ids = train_ds["doc_id"] + val_ds["doc_id"]

chunk_counts = Counter(all_doc_ids)           # {doc_id: liczba_chunków}

# przerabiamy na DataFrame (wygodne do sortowania / plotly)
df_chunks = (
    pd.Series(chunk_counts)
      .value_counts()                         # ile dokumentów ma N chunków
      .sort_index()
      .reset_index()
      .rename(columns={"index": "chunks_per_doc", 0: "n_documents"})
)

display(df_chunks.head())                     # podgląd tabeli


,chunks_per_doc,count
0,1,1
1,3,4
2,4,1
3,5,7
4,6,8


In [20]:
fig = px.bar(
    df_chunks,
    x="chunks_per_doc",
    y="count",                   # ← poprawiona nazwa kolumny
    labels={"chunks_per_doc": "Liczba chunków w dokumencie",
            "count": "Liczba dokumentów"},
    title="Rozkład liczby chunków na dokument (po flattenie)",
)
fig.show()


In [21]:
import numpy as np

# labels to lista [n_samples, n_labels]
labels_matrix = np.array(train_ds["labels"] + val_ds["labels"])   # (N, 7)

label_sums = labels_matrix.sum(axis=0)        # ile pozytywnych w każdej etykiecie
df_labels = (
    pd.DataFrame({"label_id": range(len(label_sums)),
                  "positive_count": label_sums.astype(int)})
      .assign(pos_ratio=lambda d: d.positive_count / len(labels_matrix))
)

display(df_labels)


,label_id,positive_count,pos_ratio
0,0,4992,0.706982
1,1,5148,0.729075
2,2,2822,0.399660
3,3,4722,0.668744
4,4,4421,0.626115
5,5,3278,0.464240
6,6,4840,0.685455


In [22]:
sample = train_ds[0]
print("doc_id:", sample["doc_id"])
print("tokens:", sample["input_ids"][:40], "…")
print("offsets:", sample["offset_mapping"][:10])


doc_id: 0
tokens: [0, 116, 14859, 85, 10623, 3357, 1279, 283, 4248, 16787, 1392, 14483, 6173, 2800, 12, 5, 122, 1483, 3984, 13237, 2232, 236, 5, 85, 10623, 1091, 5719, 12, 5, 205, 35058, 5, 69, 244, 9605, 126, 853, 4231, 12, 5] …
offsets: [[0, 0], [0, 1], [2, 8], [9, 10], [10, 18], [18, 23], [24, 31], [32, 34], [34, 36], [37, 41]]
